# Meeting Intelligence Demo

Run the trained system on one AMI meeting and display a complete, demo-ready result.

## Recommended output

Do **not** show only a generated paragraph. The system's value is the combination of:

1. **Narrative summary** — written by your fine-tuned BART model
2. **Key decisions** — detected by the decision classifier, with confidence and source evidence
3. **Action items** — extracted by the BIO action model as task, assignee, and deadline
4. **Selected evidence** — the important transcript chunks actually supplied to BART

The evidence section makes the result explainable: every generated summary is traceable to the
selected messages, decisions, and action items.

## 1. Setup and model paths

In [ ]:
import json
import pickle
import re
import subprocess
import sys
import warnings
import xml.etree.ElementTree as ET
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "transformers>=4.40", "datasets", "scipy", "scikit-learn", "sentencepiece"],
        check=True,
    )

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display
from scipy.sparse import csr_matrix, hstack
from sklearn.preprocessing import StandardScaler
from transformers import (
    AutoModelForSequenceClassification,
    AutoModelForSeq2SeqLM,
    AutoModelForTokenClassification,
    AutoTokenizer,
)

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def find_root():
    for candidate in [
        Path("."), Path(".."), Path("/content"), Path("/content/project"),
        Path("/content/drive/MyDrive/Graduation Project"),
    ]:
        if (candidate / "amicorpus" / "abstractive").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find amicorpus/abstractive. Set ROOT to the project folder."
    )

ROOT = find_root()
AMI_ROOT = ROOT / "amicorpus"
MODELS_DIR = ROOT / "models"

IMPORTANCE_PATH = MODELS_DIR / "importance_classifier.pkl"
DECISION_PATH = MODELS_DIR / "bert_decision"
ACTION_PATH = MODELS_DIR / "bert_action_extractor"

# Leave as None to use the standard path. If your Colab saved the model
# elsewhere, paste that folder here instead.
SUMMARIZER_PATH = None
if SUMMARIZER_PATH is None:
    SUMMARIZER_PATH = MODELS_DIR / "summarization" / "classifier_guided_bart_ami"

# Trained in 07_bullet_fusion.ipynb. Rewrites grouped flagged sentences into one clean
# bullet each, instead of showing the raw disconnected classifier output. Optional: if
# missing, the notebook falls back to displaying the raw flagged sentences.
FUSION_PATH = MODELS_DIR / "bullet_fusion_t5" / "bullet_fusion_t5"

# Choose any AMI meeting with an .abssumm.xml and corresponding word files.
# Use "ami" for a local AMI meeting or "meetingbank" for one external example.
INPUT_SOURCE = "ami"
MEETING_ID = "ES2002a"
MEETINGBANK_INDEX = 0  # Change this to try another external MeetingBank segment.
MAX_SOURCE_TOKENS = 900
CLASSIFIER_BATCH_SIZE = 32

missing = [p for p in [IMPORTANCE_PATH, DECISION_PATH, ACTION_PATH, SUMMARIZER_PATH] if not p.exists()]
if missing:
    hint = (
        "\n\nThe fine-tuned summarizer is created only after this save code runs:\n"
        "trainer.save_model(SAVE_PATH)\nsource_tokenizer.save_pretrained(SAVE_PATH)"
        if SUMMARIZER_PATH in missing else ""
    )
    raise FileNotFoundError("Missing trained artifact(s):\n" + "\n".join(f"  - {p}" for p in missing) + hint)

HAS_FUSION_MODEL = FUSION_PATH.exists()
if not HAS_FUSION_MODEL:
    print(f"Note: bullet fusion model not found at {FUSION_PATH}.")
    print("Run 07_bullet_fusion.ipynb to train it. Falling back to raw flagged sentences for now.")

print("Device:", DEVICE)
print("Input source:", INPUT_SOURCE)
print("Summarizer:", SUMMARIZER_PATH)

## 2. Load trained models

In [ ]:
with open(IMPORTANCE_PATH, "rb") as f:
    importance_artifact = pickle.load(f)
importance_model = importance_artifact["model"]
importance_vectorizer = importance_artifact["tfidf_vectorizer"]

decision_tokenizer = AutoTokenizer.from_pretrained(DECISION_PATH)
decision_model = AutoModelForSequenceClassification.from_pretrained(DECISION_PATH).to(DEVICE).eval()

action_tokenizer = AutoTokenizer.from_pretrained(ACTION_PATH)
action_model = AutoModelForTokenClassification.from_pretrained(ACTION_PATH).to(DEVICE).eval()
action_id2label = {int(key): value for key, value in action_model.config.id2label.items()}

summary_tokenizer = AutoTokenizer.from_pretrained(SUMMARIZER_PATH)
summary_model = AutoModelForSeq2SeqLM.from_pretrained(SUMMARIZER_PATH).to(DEVICE).eval()

if HAS_FUSION_MODEL:
    fusion_tokenizer = AutoTokenizer.from_pretrained(FUSION_PATH)
    fusion_model = AutoModelForSeq2SeqLM.from_pretrained(FUSION_PATH).to(DEVICE).eval()
else:
    fusion_tokenizer = fusion_model = None

print("Importance model:", importance_artifact["best_model_name"])
print("Decision model  :", decision_model.config._name_or_path)
print("Action model    :", action_model.config._name_or_path)
print("Summary model   :", summary_model.config._name_or_path)
print("Fusion model    :", fusion_model.config._name_or_path if fusion_model else "(not loaded)")

## 3. Parse one meeting transcript

In [ ]:
def local_name(element):
    return element.tag.rsplit("}", 1)[-1]

def parse_word_file(path):
    root = ET.parse(path).getroot()
    return [
        (element.text or "").strip()
        for element in root.iter()
        if local_name(element) == "w" and (element.text or "").strip()
    ]

def chunk_words(words, max_words=40):
    """Reconstruct sentence-like chunks from AMI word XML."""
    chunks, current, lexical_count = [], [], 0
    for word in words:
        current.append(word)
        lexical_count += bool(re.search(r"[A-Za-z0-9]", word))
        if word in {".", "?", "!"} or lexical_count >= max_words:
            text = " ".join(current)
            text = (text.replace(" ,", ",").replace(" .", ".")
                        .replace(" ?", "?").replace(" !", "!"))
            if len(text.split()) >= 3:
                chunks.append(text)
            current, lexical_count = [], 0
    if len(current) >= 3:
        chunks.append(" ".join(current))
    return chunks

def parse_meeting(meeting_id):
    utterances = []
    word_files = sorted((AMI_ROOT / "words").glob(f"{meeting_id}.*.words.xml"))
    if not word_files:
        raise FileNotFoundError(f"No word XML files found for {meeting_id}")
    for word_file in word_files:
        speaker = word_file.name.split(".")[1]
        utterances.extend(
            {"speaker": speaker, "text": text}
            for text in chunk_words(parse_word_file(word_file))
        )
    return utterances

def chunk_external_text(text, max_words=40):
    """Split an external transcript into sentence-like classifier chunks."""
    sentences = re.split(r"(?<=[.!?])\s+", text.replace("\n", " "))
    chunks, current = [], []
    for sentence in sentences:
        words = sentence.split()
        while words:
            available = max_words - len(current)
            current.extend(words[:available])
            words = words[available:]
            if len(current) >= max_words or not words:
                if len(current) >= 3:
                    chunks.append(" ".join(current))
                current = []
    return chunks

if INPUT_SOURCE == "ami":
    source_id = MEETING_ID
    utterances = parse_meeting(MEETING_ID)
elif INPUT_SOURCE == "meetingbank":
    from datasets import load_dataset
    meetingbank = load_dataset("huuuyeah/meetingbank")
    split_name = "test" if "test" in meetingbank else list(meetingbank.keys())[0]
    example = meetingbank[split_name][MEETINGBANK_INDEX]
    source_id = str(example.get("uid", example.get("id", f"meetingbank_{MEETINGBANK_INDEX}")))
    utterances = [
        {"speaker": "MeetingBank", "text": text}
        for text in chunk_external_text(example["transcript"])
    ]
    print("Reference summary (not used by the model):\n", example.get("summary", "(not provided)"))
else:
    raise ValueError("INPUT_SOURCE must be 'ami' or 'meetingbank'.")

print(f"Transcript chunks: {len(utterances)}")
display(pd.DataFrame(utterances).head(10))

## 4. Run the importance, decision, and action models

In [ ]:
# Same importance features used by notebooks/02_importance_classifier.ipynb.
MODAL_VERBS = re.compile(r"\b(will|would|shall|should|must|need to|ought to|going to|have to|need|wants to)\b", re.I)
DECISION_KWS = re.compile(r"\b(decided|agreed|confirmed|resolved|concluded|chosen|approved|rejected|accepted|finalised|finalized|going with|stick with|settle on)\b", re.I)
ACTION_VERBS = re.compile(r"\b(make|create|build|prepare|write|send|review|check|update|implement|design|discuss|look into|find out|contact|set up|organise|organize|schedule|arrange|follow up)\b", re.I)
NEGATION_KWS = re.compile(r"\b(not|no|never|without|don't|doesn't|didn't|won't|wouldn't|can't|cannot)\b", re.I)
FIRST_PERSON = re.compile(r"\b(I|we|our|my|us)\b")
QUESTION_RE = re.compile(r"\?")

def importance_features(texts):
    rows = []
    for text in texts:
        words = text.split()
        n = max(len(words), 1)
        rows.append([
            len(MODAL_VERBS.findall(text)), len(DECISION_KWS.findall(text)),
            len(ACTION_VERBS.findall(text)), len(NEGATION_KWS.findall(text)),
            len(FIRST_PERSON.findall(text)), int(bool(QUESTION_RE.search(text))),
            len(words), len(text), len(MODAL_VERBS.findall(text)) / n,
            sum(word[0].isupper() and len(word) > 1 for word in words) / n,
            int(text.endswith(".")), text.count(",") / n,
        ])
    return np.asarray(rows, dtype=float)

def score_importance(texts):
    x_tfidf = importance_vectorizer.transform(texts)
    x_handcrafted = StandardScaler().fit_transform(importance_features(texts))
    x = hstack([x_tfidf, csr_matrix(x_handcrafted)])
    return importance_model.predict_proba(x)[:, 1]

@torch.no_grad()
def score_decisions(texts):
    values = []
    for start in range(0, len(texts), CLASSIFIER_BATCH_SIZE):
        batch = texts[start:start + CLASSIFIER_BATCH_SIZE]
        encoded = decision_tokenizer(batch, truncation=True, padding=True,
                                     max_length=128, return_tensors="pt").to(DEVICE)
        values.extend(torch.softmax(decision_model(**encoded).logits, dim=-1)[:, 1].cpu().tolist())
    return np.asarray(values)

def decode_bio(tokens, labels):
    spans, start, entity = [], None, None
    for index, label in enumerate(labels + ["O"]):
        if label.startswith("B-"):
            if entity is not None:
                spans.append((entity, start, index))
            entity, start = label[2:], index
        elif label.startswith("I-") and entity == label[2:]:
            continue
        else:
            if entity is not None:
                spans.append((entity, start, index))
            entity, start = None, None
    return spans

@torch.no_grad()
def extract_action_items(texts):
    """Return one list of {action, assignee, deadline} dictionaries per text."""
    outputs = []
    for start in range(0, len(texts), CLASSIFIER_BATCH_SIZE):
        batch = texts[start:start + CLASSIFIER_BATCH_SIZE]
        token_lists = [text.split() for text in batch]
        encoded = action_tokenizer(token_lists, is_split_into_words=True, truncation=True,
                                   padding=True, max_length=128, return_tensors="pt").to(DEVICE)
        predictions = action_model(**encoded).logits.argmax(dim=-1).cpu().numpy()
        for row, tokens in enumerate(token_lists):
            labels, seen = [], set()
            for position, word_id in enumerate(encoded.word_ids(batch_index=row)):
                if word_id is not None and word_id not in seen:
                    seen.add(word_id)
                    labels.append(action_id2label[int(predictions[row, position])])
            spans = decode_bio(tokens, labels[:len(tokens)])
            by_type = {}
            for entity, span_start, span_end in spans:
                by_type.setdefault(entity, []).append(" ".join(tokens[span_start:span_end]))
            outputs.append([
                {
                    "action": action,
                    "assignee": (by_type.get("ASSIGNEE") or [None])[0],
                    "deadline": (by_type.get("DEADLINE") or [None])[0],
                }
                for action in by_type.get("ACTION", [])
            ])
    return outputs

texts = [row["text"] for row in utterances]
importance_scores = score_importance(texts)
decision_scores = score_decisions(texts)
action_items = extract_action_items(texts)

for index, row in enumerate(utterances):
    row["index"] = index  # original chronological position, used later to group adjacent evidence
    row["importance"] = float(importance_scores[index])
    row["decision_score"] = float(decision_scores[index])
    row["is_decision"] = bool(decision_scores[index] >= 0.50)
    row["action_items"] = action_items[index]
    row["is_action"] = bool(action_items[index])

print("Decisions:", sum(row["is_decision"] for row in utterances))
print("Action-bearing chunks:", sum(row["is_action"] for row in utterances))

## 5. Select evidence and generate the narrative summary

In [ ]:
def build_guided_input(rows):
    ranked = sorted(
        range(len(rows)),
        key=lambda i: (rows[i]["is_action"] or rows[i]["is_decision"], rows[i]["importance"]),
        reverse=True,
    )
    selected, used_tokens = [], 0
    for index in ranked:
        row = rows[index]
        marker = ("[DECISION] " if row["is_decision"] else "") + ("[ACTION] " if row["is_action"] else "")
        line = f"{marker}{row['speaker']}: {row['text']}"
        line_tokens = len(summary_tokenizer(line, add_special_tokens=False).input_ids)
        if used_tokens + line_tokens <= MAX_SOURCE_TOKENS:
            selected.append((index, line))
            used_tokens += line_tokens
    selected.sort(key=lambda pair: pair[0])  # chronological reading order
    return "\n".join(line for _, line in selected), [index for index, _ in selected]

guided_input, selected_indices = build_guided_input(utterances)
encoded = summary_tokenizer(guided_input, return_tensors="pt", truncation=True,
                            max_length=MAX_SOURCE_TOKENS).to(DEVICE)
with torch.no_grad():
    generated = summary_model.generate(
        **encoded,
        num_beams=4,
        length_penalty=2.0,
        min_new_tokens=45,
        max_new_tokens=220,
    )
summary = summary_tokenizer.decode(generated[0], skip_special_tokens=True)

selected_rows = [utterances[index] for index in selected_indices]
print(f"Selected evidence: {len(selected_rows)}/{len(utterances)} chunks")
print(f"Input tokens: {len(encoded['input_ids'][0])}/{MAX_SOURCE_TOKENS}")
display(Markdown(f"## Generated narrative summary\n\n{summary}"))

## 5b. Group adjacent evidence and fuse into clean bullets

The decision/action classifiers flag individual sentences, which reads as disconnected fragments.
Consecutive flagged sentences (by original transcript position) are grouped into one evidence group,
then rewritten into a single bullet by the fusion model trained in `07_bullet_fusion.ipynb`. For
actions, only the action phrase is rewritten — assignee and deadline come straight from the BIO
tagger so the fusion model can't invent a person or date that was never said.

In [ ]:
FUSION_MAX_SOURCE_TOKENS = 160
FUSION_MAX_TARGET_TOKENS = 64
GROUP_MAX_INDEX_GAP = 2  # merge flagged rows into one group if they're this close in the transcript

@torch.no_grad()
def fuse_bullets(evidence_sentences, task_type="decision"):
    """Rewrite a list of noisy flagged sentences into one clean bullet. Mirrors 07_bullet_fusion.ipynb."""
    if not evidence_sentences:
        return ""
    prefixed = f"fuse {task_type}: " + " ".join(evidence_sentences)
    encoded = fusion_tokenizer(prefixed, truncation=True, max_length=FUSION_MAX_SOURCE_TOKENS,
                               return_tensors="pt").to(DEVICE)
    generated = fusion_model.generate(**encoded, num_beams=4, max_new_tokens=FUSION_MAX_TARGET_TOKENS)
    return fusion_tokenizer.decode(generated[0], skip_special_tokens=True)

def group_consecutive(rows, max_gap=GROUP_MAX_INDEX_GAP):
    """Group rows whose original transcript index is within max_gap of the previous row in the group."""
    groups, current = [], []
    for row in sorted(rows, key=lambda r: r["index"]):
        if current and row["index"] - current[-1]["index"] > max_gap:
            groups.append(current)
            current = []
        current.append(row)
    if current:
        groups.append(current)
    return groups

def build_bullet(group, task_type):
    evidence_sentences = [f"{row['speaker']}: {row['text']}" for row in group]
    if HAS_FUSION_MODEL:
        return fuse_bullets(evidence_sentences, task_type=task_type)
    return " / ".join(row["text"] for row in group)  # fallback: raw sentences, joined

decision_groups = group_consecutive([row for row in selected_rows if row["is_decision"]])
action_groups = group_consecutive([row for row in selected_rows if row["action_items"]])

decision_bullets = [
    {
        "bullet": build_bullet(group, "decision"),
        "confidence": max(row["decision_score"] for row in group),
        "speakers": ", ".join(dict.fromkeys(row["speaker"] for row in group)),
        "n_evidence": len(group),
        "evidence": " | ".join(row["text"] for row in group),
    }
    for group in decision_groups
]

action_bullets = []
for group in action_groups:
    assignees = [item["assignee"] for row in group for item in row["action_items"] if item["assignee"]]
    deadlines = [item["deadline"] for row in group for item in row["action_items"] if item["deadline"]]
    action_bullets.append({
        "bullet": build_bullet(group, "action"),
        "assignee": ", ".join(dict.fromkeys(assignees)) or "\u2014",
        "deadline": ", ".join(dict.fromkeys(deadlines)) or "\u2014",
        "speakers": ", ".join(dict.fromkeys(row["speaker"] for row in group)),
        "n_evidence": len(group),
        "evidence": " | ".join(row["text"] for row in group),
    })

print(f"Decision groups: {len(decision_bullets)}  |  Action groups: {len(action_bullets)}")
print("Bullet fusion model:", "active" if HAS_FUSION_MODEL else "not available, showing raw sentences")

## 6. Display meeting intelligence

In [ ]:
def bullet(text):
    return text.replace("\n", " ").strip()

display(Markdown("## Key decisions"))
if decision_bullets:
    for row in decision_bullets:
        display(Markdown(
            f"- {bullet(row['bullet'])}  \n"
            f"  _Confidence: {row['confidence']:.0%} · Speakers: {row['speakers']} · "
            f"Fused from {row['n_evidence']} utterance(s)_"
        ))
else:
    display(Markdown("_No decision was detected in the selected evidence._"))

display(Markdown("## Action items"))
if action_bullets:
    display(pd.DataFrame([
        {
            "Action": row["bullet"],
            "Assignee": row["assignee"],
            "Deadline": row["deadline"],
            "Speakers": row["speakers"],
        }
        for row in action_bullets
    ]))
else:
    display(Markdown("_No action item was detected in the selected evidence._"))

action_table = [  # kept for the JSON export in the next section
    {"Action": row["bullet"], "Assignee": row["assignee"], "Deadline": row["deadline"],
     "Speakers": row["speakers"], "Evidence": row["evidence"]}
    for row in action_bullets
]

## 7. Evidence and export

In [ ]:
evidence = pd.DataFrame([
    {
        "speaker": row["speaker"],
        "text": row["text"],
        "importance": round(row["importance"], 3),
        "decision_score": round(row["decision_score"], 3),
        "decision": row["is_decision"],
        "action": row["is_action"],
    }
    for row in selected_rows
]).sort_values("importance", ascending=False)

display(Markdown("## Selected evidence supplied to the summarizer"))
display(evidence)

result = {
    "meeting_id": source_id,
    "summary": summary,
    "key_decisions": [
        {"bullet": row["bullet"], "confidence": row["confidence"],
         "speakers": row["speakers"], "evidence": row["evidence"]}
        for row in decision_bullets
    ],
    "action_items": action_table,
    "selected_evidence": evidence.to_dict(orient="records"),
}

safe_id = re.sub(r"[^A-Za-z0-9_-]+", "_", source_id)
output_path = ROOT / "results" / f"{safe_id}_meeting_intelligence.json"
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Saved demo result → {output_path}")